<a href="https://colab.research.google.com/github/jiyuutheosum/Machine-Learning/blob/main/Machine_Learning_PIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name: [ Jalanie Baraocor ] [ Sherri Nicole Tilan ]**

**BSIT - 4R8 - Machine Learning**

# **1. Installing necessary tools**

In [17]:
# === Cell 1: Install & downloads (run once in Colab) ===
!pip install -q pyspellchecker
!pip install -q scikit-learn
!pip install -q transformers datasets evaluate
!pip install shap

# NLTK downloads
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# **2. Importing libraries**

In [10]:
# === Cell 2: Imports & load dataset ===
import os, re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from spellchecker import SpellChecker
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import joblib
from google.colab import files
import pandas as pd
import io

# **3. Load Dataset**

In [11]:
print("Upload your dataset (.xlsx or .csv)")
uploaded = files.upload()   # Opens the file picker UI

# Automatically detect the uploaded file name
file_name = next(iter(uploaded))

# Load Excel or CSV
if file_name.endswith(".xlsx"):
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))
elif file_name.endswith(".csv"):
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
else:
    raise ValueError("Unsupported file type. Please upload .xlsx or .csv")

print("\n✔ Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Upload your dataset (.xlsx or .csv)


Saving full-translated-to-eng-dataset.xlsx to full-translated-to-eng-dataset.xlsx

✔ Dataset loaded successfully!
Shape: (16175, 3)


,sents,sentiments,topics
0,Full course slides .,2,1
1,"Enthusiastic teaching, close to students.",2,0
2,Attend school with full marks for attendance.,0,1
3,have not yet applied information technology an...,0,0
4,"The teacher teaches well, there are many examp...",2,0


# **4. Label Mapping**

In [18]:
import pandas as pd
import numpy as np

# ---- Auto-detect TEXT column ----
text_candidates = [
    c for c in df.columns
    if any(x in c.lower() for x in ['feedback','comment','text','review','sents','response'])
]

if not text_candidates:
    raise ValueError("No text column found. Please rename your text column to something like 'feedback' or 'comment'.")

text_col = text_candidates[0]

# ---- Auto-detect LABEL column ----
label_candidates = [
    c for c in df.columns
    if any(x in c.lower() for x in ['label','sentiment','rating','class'])
]

# fallback: find binary-like columns
if not label_candidates:
    for c in df.columns:
        if df[c].nunique() == 2 and c != text_col:
            label_candidates.append(c)

if not label_candidates:
    raise ValueError("No label/sentiment column found. Please rename your label column.")

label_col = label_candidates[0]

print(f"Detected text column: {text_col}")
print(f"Detected label column: {label_col}")

# ---- Build the cleaned df ----
df = df[[text_col, label_col]].rename(columns={
    text_col: "text",
    label_col: "original_label"
}).dropna().reset_index(drop=True)

# ---- Sample only 5000 rows ----
SAMPLE_N = 5000
if len(df) > SAMPLE_N:
    df = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)

print("Sampled shape:", df.shape)

# ---- Label Mapping ----
def map_label(val):
    v = str(val).strip().lower()

    positive_words = ['positive','pos','p','1','yes','y','good','happy']
    negative_words = ['negative','neg','n','0','no','bad','angry']

    # String label mapping
    if v in positive_words:
        return 1
    if v in negative_words:
        return 0

    # Numeric label mapping
    if v.isdigit():
        num = int(v)
        if num == 1: return 1
        if num == 0: return 0
        if 1 <= num <= 5:
            return 1 if num >= 3 else 0

    # fallback
    return 1 if 'pos' in v or '1' in v else 0

df["label"] = df["original_label"].apply(map_label)

print("Label distribution:")
print(df["label"].value_counts())


Detected text column: text
Detected label column: original_label
Sampled shape: (5000, 2)
Label distribution:
label
0    4777
1     223
Name: count, dtype: int64


# **5. Data Cleaning**

In [19]:
# === Preprocessing pipeline (cached spellchecker) ===
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
spell = SpellChecker(distance=1)
spell_cache = {}

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # reduce repeated chars
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    tokens = word_tokenize(text)
    cleaned = []
    for t in tokens:
        if t in stop_words:
            continue
        if len(t) > 2:
            if t in spell_cache:
                t = spell_cache[t]
            else:
                corr = spell.correction(t)
                if corr:
                    spell_cache[t] = corr
                    t = corr
        t = lemmatizer.lemmatize(t)
        cleaned.append(t)
    return " ".join(cleaned)

# Apply
df['clean_text'] = df[text_col].astype(str).map(clean_text)
display(df[['clean_text']].head(8))

,clean_text
0,teacher came place lecture understand
1,advanced knowledge teacher presented still dif...
2,need teacher develop specific set textbook doc...
3,good exercise
4,enthusiastic teacher thank teacher
5,dedicated enthusiastic teacher
6,teacher still bit difficult understand
7,teacher gave score bit tight


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('clean_text').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['index']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'index'}, axis=1)
              .sort_values('index', ascending=True))
  xs = counted['index']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_2.sort_values('index', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('clean_text')):
  _plot_series(series, series_name, i)
  fig.legend(title='clean_text', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_3['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_4['clean_text'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_4, x='index', y='clean_text', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

# **5. Splitting**

In [20]:
# === split into train/val/test (70/15/15 stratified) ===
X = df['clean_text'].values
y = df['label'].values
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
val_relative = 0.15 / (1 - 0.15)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=val_relative, random_state=42, stratify=y_train_val)
print("Sizes: train", len(X_train), "val", len(X_val), "test", len(X_test))

Sizes: train 3500 val 750 test 750


# **6. Feature Extraction TF-IDF & BoW**

In [31]:
# === feature extraction (TF-IDF & BoW) ===
bow = CountVectorizer(ngram_range=(1,1), max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_val_bow = bow.transform(X_val)
X_test_bow = bow.transform(X_test)

corpus = df['clean_text'].fillna('').tolist()
X_counts = bow.fit_transform(corpus)  # sparse matrix (n_docs x n_features)
print("BoW shape:", X_counts.shape)

# Convert to DataFrame (feature names as columns)
bow_df = pd.DataFrame(X_counts.toarray(), columns=bow.get_feature_names_out())
bow_df.index = df.index  # align indices with original df
bow_df.head()

BoW shape: (5000, 2536)


,00,10,100,10th,11,11doubledot55,12,12doubledot00,13,13h00,...,wzjwz83,wzjwz9,wzjwz92,wzjwz94,xml,year,yes,yet,young,zero
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [35]:
tfidf = TfidfVectorizer(ngram_range=(1,1), max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

X_tfidf = tfidf.fit_transform(corpus)
print("TF-IDF shape:", X_tfidf.shape)

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
tfidf_df.index = df.index
tfidf_df.head()

TF-IDF shape: (5000, 2536)


,00,10,100,10th,11,11doubledot55,12,12doubledot00,13,13h00,...,wzjwz83,wzjwz9,wzjwz92,wzjwz94,xml,year,yes,yet,young,zero
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# **7. Train Classical Models and Evaluate**

In [36]:
# === train classical models (fast) & evaluate on validation set ===
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC(max_iter=10000),
    "MultinomialNB": MultinomialNB(),
    "RidgeClassifier": RidgeClassifier(),
    "PassiveAggressive": PassiveAggressiveClassifier(max_iter=1000)
}

results = []
def evaluate(name, model, Xtr, Xv, ytr, yv):
    model.fit(Xtr, ytr)
    preds = model.predict(Xv)
    acc = accuracy_score(yv, preds)
    prec = precision_score(yv, preds, zero_division=0)
    rec = recall_score(yv, preds, zero_division=0)
    f1 = f1_score(yv, preds, zero_division=0)
    cm = confusion_matrix(yv, preds)
    print(f"\n{name}: acc={acc:.4f}, prec={prec:.4f}, rec={rec:.4f}, f1={f1:.4f}")
    print("confusion:\n", cm)
    print(classification_report(yv, preds, zero_division=0))
    results.append({'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'cm': cm})

print("=== TF-IDF models ===")
for n,m in models.items():
    evaluate(n + " (tfidf)", m, X_train_tfidf, X_val_tfidf, y_train, y_val)

print("\n=== BoW models ===")
for n,m in models.items():
    # reinstantiate to avoid warm state issues
    if n=="LogisticRegression": mm=LogisticRegression(max_iter=1000)
    elif n=="LinearSVC": mm=LinearSVC(max_iter=10000)
    elif n=="MultinomialNB": mm=MultinomialNB()
    elif n=="RidgeClassifier": mm=RidgeClassifier()
    else: mm=PassiveAggressiveClassifier(max_iter=1000)
    evaluate(n + " (bow)", mm, X_train_bow, X_val_bow, y_train, y_val)

=== TF-IDF models ===

LogisticRegression (tfidf): acc=0.9547, prec=0.0000, rec=0.0000, f1=0.0000
confusion:
 [[716   0]
 [ 34   0]]
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       716
           1       0.00      0.00      0.00        34

    accuracy                           0.95       750
   macro avg       0.48      0.50      0.49       750
weighted avg       0.91      0.95      0.93       750


LinearSVC (tfidf): acc=0.9533, prec=0.4286, rec=0.0882, f1=0.1463
confusion:
 [[712   4]
 [ 31   3]]
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       716
           1       0.43      0.09      0.15        34

    accuracy                           0.95       750
   macro avg       0.69      0.54      0.56       750
weighted avg       0.93      0.95      0.94       750


MultinomialNB (tfidf): acc=0.9547, prec=0.0000, rec=0.0000, f1=0.0000
confusion:
 [[716   0]
 [ 34   0]]
     

# **9. Picking the best Model**

In [37]:
# === pick best TF-IDF model by F1, retrain on train+val, evaluate on test & save artifacts ===
import numpy as np
tfidf_results = [r for r in results if '(tfidf)' in r['model']]
best = sorted(tfidf_results, key=lambda x: x['f1'], reverse=True)[0]
print("Best on validation (tfidf):", best['model'])

best_short = best['model'].split()[0]
if best_short == "LogisticRegression":
    final_model = LogisticRegression(max_iter=1000)
elif best_short == "LinearSVC":
    final_model = LinearSVC(max_iter=10000)
elif best_short == "MultinomialNB":
    final_model = MultinomialNB()
elif best_short == "RidgeClassifier":
    final_model = RidgeClassifier()
else:
    final_model = PassiveAggressiveClassifier(max_iter=1000)

from scipy.sparse import vstack
X_train_val_tfidf = vstack([X_train_tfidf, X_val_tfidf])
y_train_val = np.concatenate([y_train, y_val])
final_model.fit(X_train_val_tfidf, y_train_val)
test_preds = final_model.predict(X_test_tfidf)

print("Final test metrics:")
print("Accuracy:", accuracy_score(y_test, test_preds))
print("Precision:", precision_score(y_test, test_preds, zero_division=0))
print("Recall:", recall_score(y_test, test_preds, zero_division=0))
print("F1:", f1_score(y_test, test_preds, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, test_preds))

# Save vectorizers and model (joblib)
os.makedirs('/drive/MyDrive/ML/models', exist_ok=True)
joblib.dump(tfidf, '/drive/MyDrive/ML/models/tfidf_vectorizer.joblib')
joblib.dump(bow, '/drive/MyDrive/ML/models/bow_vectorizer.joblib')
joblib.dump(final_model, '/drive/MyDrive/ML/models/final_model.joblib')
print("Saved artifacts to /drive/Mydrive/ML/models")

Best on validation (tfidf): RidgeClassifier (tfidf)
Final test metrics:
Accuracy: 0.9573333333333334
Precision: 0.6666666666666666
Recall: 0.06060606060606061
F1: 0.1111111111111111
Confusion matrix:
 [[716   1]
 [ 31   2]]
Saved artifacts to /drive/Mydrive/ML/models


# **10. XAI SHAP on Classical Models**

In [44]:
import shap
import numpy as np

# Use a small background sample for SHAP (recommended)
background = X_train_tfidf[np.random.choice(X_train_tfidf.shape[0], 200, replace=False)]

# Kernel SHAP for text ML models
explainer = shap.KernelExplainer(model.predict_proba, background)

# Choose sample text
sample_text = ["the instructor was very helpful and the module was easy to follow"]

# Convert text to vector
sample_vector = tfidf.transform(sample_text)

# Compute SHAP values
shap_values = explainer.shap_values(sample_vector)

# Visualize
shap.initjs()
shap.force_plot(explainer.expected_value[1], shap_values[1], feature_names=tfidf.get_feature_names_out())

NameError: name 'model' is not defined

# **11. Deep Learning**

In [ ]:
# Training LSTM / CNN (use GPU runtime in Colab)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
tokenizer = Tokenizer(num_words=20000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
X_tr_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)
maxlen=200
X_tr_pad = pad_sequences(X_tr_seq, maxlen=maxlen)
X_val_pad = pad_sequences(X_val_seq, maxlen=maxlen)
model = Sequential([
    Embedding(20000, 128, input_length=maxlen),
    LSTM(128),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_tr_pad, y_train, validation_data=(X_val_pad, y_val), epochs=5, batch_size=64)

In [ ]:
# Training DistilBERT (huggingface + tensorflow)

from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification
import tensorflow as tf

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_enc = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
val_enc = tokenizer(list(X_val), truncation=True, padding=True, max_length=128)

train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_enc), y_train)).shuffle(1000).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_enc), y_val)).batch(16)

model = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
model.compile(optimizer=optimizer, loss=model.compute_loss, metrics=['accuracy'])
model.fit(train_dataset, validation_data=val_dataset, epochs=3)

# **XAI SHAP for LSTM**

In [ ]:
import shap
import numpy as np
import tensorflow as tf

# Create SHAP explainer for deep learning
explainer = shap.DeepExplainer(model, X_train_pad[:200])

# Pick sample input (already tokenized & padded)
sample_input = X_test_pad[:1]

# Compute SHAP values
shap_values = explainer.shap_values(sample_input)

# Plot
shap.image_plot(shap_values)

# **XAI SHAP for DistilBERT**

In [ ]:
import shap
import transformers
import torch

tokenizer = transformers.DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Wrap model to probability output
def predict_proba(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).numpy()
    return probs

# Create SHAP explainer
explainer = shap.Explainer(predict_proba, tokenizer)

# Sample text
sample_text = ["the lesson was confusing but the instructor explained it well"]

# Calculate SHAP values
shap_values = explainer(sample_text)

# Plot
shap.plots.text(shap_values[0])